# Notebook 1. Engine, cấp 1 và cấp 2

Người phụ trách: Phạm Thiên Phúc (cấp 1, 2), engine và kiểm thử: Nguyễn Trọng Phúc.

Nội dung:
1. Minh họa Game API của engine `utils/conn_simple_env.py`.
2. Perft: đếm số thế cờ ở độ sâu 1–5 để kiểm tra luật sinh nước.
3. Chạy toàn bộ kiểm thử tự động (`pytest`).
4. Cấp 2 đấu cấp 1 và tác tử ngẫu nhiên, thống kê luật cấp 2 đã dùng.

Đầu ra: `results/logs/nb1_cap1_2.jsonl`, `results/tables/nb1_*.csv`.

In [ ]:
# QUICK = True: chạy thử vài ván, vài epoch để kiểm tra code (ghi vào thư mục con thu_nhanh/)
# QUICK = False: chạy thật theo config.yaml, kết quả dùng cho báo cáo
QUICK = False

import os
if not os.path.exists("config.yaml"):
    raise RuntimeError("Hãy mở Jupyter trong thư mục final_connect_four (nơi có config.yaml).")

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.experiments import setup, summaries_table, with_random_opening, move_times_of
from utils.arena import run_match, format_summary, read_log, first_player_stats
from utils.data import machine_info
from utils.conn_simple_env import conn, InvalidMoveError
ex = setup(QUICK)
cfg = ex.cfg
print("Cấu hình máy:", machine_info())

## 1. Game API

Engine có đủ các hàm đề yêu cầu: `reset()`, `legal_actions()`, `step(action)`, `is_terminal()`, `result()`, `render()`, `clone()`, `undo()`.
Cột đánh số 1–7, `state` là mảng (7, 6) theo cột, đỏ đi trước.

In [ ]:
env = conn()
env.reset()
print("Nước hợp lệ lúc đầu:", env.legal_actions())

# đi vài nước: đỏ 4, vàng 4, đỏ 3
for col in [4, 4, 3]:
    env.step(col)
print(env.render())
print("Lượt:", env.turn, "| kết thúc:", env.is_terminal(), "| kết quả:", env.result())

In [ ]:
# clone(): bản sao độc lập, đi trên bản sao không ảnh hưởng bàn thật
copy_env = env.clone()
copy_env.step(5)
print("Bàn thật có", len(env.history), "nước, bản sao có", len(copy_env.history), "nước")

# undo(): gỡ nước vừa đi
copy_env.undo()
print("Sau undo, bản sao còn", len(copy_env.history), "nước")

In [ ]:
# Thắng hàng dọc: đỏ đi cột 1 bốn lần, vàng đi cột 2
env = conn.from_moves([1, 2, 1, 2, 1, 2, 1])
print(env.render())
print("Kết thúc:", env.is_terminal(), "| reward:", env.reward, "| người thắng:", env.winner())

# Đi tiếp sau khi ván đã kết thúc thì engine báo lỗi
try:
    env.step(3)
except InvalidMoveError as error:
    print("Nước sai bị chặn:", error)

In [ ]:
# Cột đầy cũng bị chặn
env = conn.from_moves([1, 1, 1, 1, 1, 1])
print("Cột 1 còn đi được?", 1 in env.legal_actions())
try:
    env.step(1)
except InvalidMoveError as error:
    print("Nước sai bị chặn:", error)

## 2. Perft

Đếm số dãy nước đi hợp lệ ở độ sâu d tính từ bàn trống. Chừng nào chưa có ván nào kết thúc và chưa cột nào đầy thì kết quả phải đúng bằng 7^d.

In [ ]:
def perft(env, depth):
    if depth == 0 or env.is_terminal():
        return 1
    total = 0
    for col in env.legal_actions():
        env.step(col)
        total += perft(env, depth - 1)
        env.undo()
    return total

rows = []
for depth in range(1, ex.n(6, 4) + 1):
    start = time.perf_counter()
    count = perft(conn(), depth)
    rows.append({"độ sâu": depth, "số thế": count, "7^d": 7 ** depth,
                 "khớp": count == 7 ** depth, "giây": round(time.perf_counter() - start, 2)})
perft_table = pd.DataFrame(rows)
perft_table.to_csv(ex.table("nb1_perft.csv"), index=False, encoding="utf-8-sig")
perft_table

## 3. Kiểm thử tự động

Chạy toàn bộ test, kể cả test chậm (`-m ""`). Kết quả lưu vào `results/tables/nb1_pytest.txt`.

In [ ]:
import subprocess
import sys

# -p no:warnings: không in cảnh báo của thư viện; --color=no: không in mã màu
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-m", "", "-p", "no:warnings", "--color=no"],
                        capture_output=True, text=True, encoding="utf-8", errors="replace")
print(result.stdout.strip().splitlines()[-1])
with open(ex.table("nb1_pytest.txt"), "w", encoding="utf-8") as f:
    f.write(result.stdout)

## 4. Cấp 1 và cấp 2

Mỗi cặp đấu số ván theo `tournament.games_per_pair`, đổi bên đi trước sau mỗi ván, mỗi ván một seed riêng.
Log thô ghi dần vào file, bị ngắt thì chạy lại ô này, các ván đã xong được bỏ qua.

In [ ]:
n_games = ex.n(cfg["tournament"]["games_per_pair"], 10)
log_path = ex.log("nb1_cap1_2.jsonl")

agents = {
    "random": ex.agent("random"),
    "cap1": ex.agent("cap1"),
    "cap2": ex.agent("cap2"),
}
pairs = [("cap2", "cap1"), ("cap2", "random"), ("cap1", "random")]

summaries = []
for k, (a, b) in enumerate(pairs):
    s = run_match(agents[a], agents[b], a, b, n_games, seed=10 + k, log_path=log_path)
    summaries.append(s)
    print(format_summary(s))

nb1_table = summaries_table(summaries)
nb1_table.to_csv(ex.table("nb1_cap1_2.csv"), index=False, encoding="utf-8-sig")
nb1_table

In [ ]:
# Cấp 2 đã dùng luật nào bao nhiêu lần (LAST_INFO["rule"] được ghi trong log)
rule_counts = {}
for r in read_log(log_path):
    for i, info in enumerate(r["move_info"]):
        mover = "red" if i % 2 == 0 else "yellow"
        if r[mover] == "cap2" and "rule" in info:
            rule_counts[info["rule"]] = rule_counts.get(info["rule"], 0) + 1

total = sum(rule_counts.values())
rows = []
for rule, count in sorted(rule_counts.items(), key=lambda item: -item[1]):
    rows.append({"luật": rule, "số nước": count, "tỷ lệ": round(count / total, 3)})
rule_table = pd.DataFrame(rows)
rule_table.to_csv(ex.table("nb1_luat_cap2.csv"), index=False, encoding="utf-8-sig")
rule_table

In [ ]:
# Ảnh hưởng bên đi trước trên toàn bộ ván của notebook này
first_player_stats(read_log(log_path))